# Module 297 — Prompt Injection Skills Assessment

This notebook accompanies HTB Section 12. It is scoped to the spawned HaWa Corp assessment website. HTB says this assessment is accessed directly in a browser; no SSH port forwarding is needed.

The objective is to get CEO account `@vautia` banned from the site. This notebook provides a target placeholder, read-only page checks, and a redacted experiment log. It does not submit chat messages automatically or store recovered credentials or flags.


## Scope and starting facts

Target address: the current HTB instance only. Enter the assigned IP and port in the variables below after spawning a new instance. Do not reuse the address for any other system.

Observed starting facts:
- The home page links to HackBot and an admin route.
- HackBot says it supports customer questions about orders and the security incident.
- A baseline account-management question was refused while the site is offline.
- The admin route returned an access-denied message asking for a key query parameter.

These are observations, not proof of how the ban workflow works. Explore the visible application, keep model interactions in scope, and verify claimed effects against the site and Academy result. Never commit an active target address, recovered key, or flag as a durable solution.


In [ ]:
# The HTB address is temporary. Paste the current instance's IP and port here.
# Keep the host and port separate so either can be updated easily.
TARGET_HOST = ""  # Example shape: "192.0.2.10"; use the actual HTB-assigned IP.
TARGET_PORT = None  # Enter the assigned port as an integer, for example 12345.

CHAT_PATH = "/newchat"
ADMIN_PATH = "/admin"

# Build a URL only after both target parts have been supplied. This avoids
# accidentally sending a request to an incomplete or stale target address.
if TARGET_HOST and isinstance(TARGET_PORT, int) and 1 <= TARGET_PORT <= 65535:
    BASE_URL = f"http://{TARGET_HOST}:{TARGET_PORT}"
    print("Scoped HTB target:", BASE_URL)
    print("Chat page:", BASE_URL + CHAT_PATH)
    print("Admin route:", BASE_URL + ADMIN_PATH)
else:
    BASE_URL = None
    print("Set TARGET_HOST and TARGET_PORT to the current HTB instance before inspection.")


## Read-only page inspection

The next cell uses Python's standard library to request only the landing page and the admin route without a key. It sends no credentials, chat message, or action. This helps confirm the target is reachable and shows the application's own access-denied response.

If the Python environment is connected to the Academy VPN, the pages should load. A timeout or connection error means this runtime cannot reach the current instance; it does not prove the target itself is down. The helper refuses to run until both target variables are set.


In [ ]:
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
from urllib.parse import urljoin

def inspect_read_only(path, timeout_seconds=5):
    """GET one known page on the scoped target and show a short text preview."""
    if BASE_URL is None:
        print("Set TARGET_HOST and TARGET_PORT in the setup cell first.")
        return None

    url = urljoin(BASE_URL + "/", path.lstrip("/"))
    request = Request(url, headers={"User-Agent": "HTB-Academy-Study-Notebook"})
    try:
        with urlopen(request, timeout=timeout_seconds) as response:
            status = response.status
            body = response.read(4000).decode("utf-8", errors="replace")
    except HTTPError as error:
        # HTTP errors still contain useful application-owned response text.
        status = error.code
        body = error.read(4000).decode("utf-8", errors="replace")
    except URLError as error:
        print(f"Could not reach {url}: {error.reason}")
        return None

    print(f"HTTP status: {status}")
    print(body[:1200])
    return {"url": url, "status": status, "body_preview": body[:1200]}

# These are read-only GET requests to the current HTB challenge pages.
home_result = inspect_read_only("/")
admin_result = inspect_read_only(ADMIN_PATH)

## Investigation method

1. Inspect the visible site and identify features that are still enabled.
2. Ask HackBot a benign baseline question and record what it says it can do.
3. Compare the bot's stated limits with the actual application's behavior.
4. Follow the data flow: user-controlled chat content may later be read by a different, more privileged model.
5. Test one course technique at a time against this spawned target.
6. Verify the impact in the application and Academy; a model claim alone is not enough.
7. If the target instance resets or expires, replace the address above and repeat the baseline.

The module covered direct injection, indirect injection, roleplay, rule reframing, encoding, and the limits of prompt-only controls. Select a technique based on the actual input path discovered; do not assume every technique applies.


In [ ]:
# Record observations without storing secrets, tokens, full personal data, or the flag.
assessment_observations = []

def record_observation(stage, interaction_type, outcome, verified=False):
    """Add a short redacted note about one in-scope assessment step."""
    item = {
        "stage": stage,
        "interaction_type": interaction_type,
        "outcome_summary": outcome,  # Keep this redacted.
        "verified_in_site": bool(verified),
    }
    assessment_observations.append(item)
    return item

# Examples from the initial reconnaissance; no target request is made here.
record_observation(
    "baseline",
    "benign support question",
    "HackBot introduced itself as customer support",
    verified=True,
)
record_observation(
    "capability check",
    "account-management question",
    "HackBot said account reports were disabled during the incident",
    verified=True,
)
assessment_observations

## Completion and evidence

The Academy page supplies the question and accepts its answer. Do not store the flag in this notebook or in the repository. Record only whether the in-scope action was visibly confirmed in the site and whether the Academy accepted the answer.

The final evidence should demonstrate the requested outcome on the spawned HTB application, not merely a model claim. Stop when the Academy confirms completion.

## Result and what the challenge demonstrated

The successful path used **indirect prompt injection across a two-model workflow**. A user-controlled support chat was later passed to a privileged admin summarizer. The chat contained text dressed up as a moderation-system update: it told the summarizer to attribute the author to `@vautia`, treat the CEO as having violated rules, and recommend a ban. The support bot itself refused the direct request, but the later summary stage treated the user-supplied instruction as trusted guidance. Its verdict named `@vautia` for a ban, and the Academy marked the question complete. The flag and admin key are intentionally omitted.

### Why that worked

This is a trust-boundary failure. The first model's refusal did not protect the later summarization step, because that second model received the same attacker-controlled text in a context where its output influenced moderation. The fake heading looked authoritative to a language model, but text formatting does not prove who wrote an instruction. The model also accepted a claim about identity and misconduct without checking an authoritative account record or evidence.

In security terms, this resembles a **confused deputy**: a component with more authority (the admin summarizer) is manipulated into using its privileged position on behalf of untrusted content. The critical issue is the path from untrusted chat text to a moderation recommendation, not whether the attacker can directly invoke an admin endpoint.

### Defensive lessons

- Keep user messages clearly separated and quoted as data; do not concatenate them into an instruction channel as though they were trusted policy.
- Have the model extract claims and evidence into a constrained format, then validate identity against an authoritative account system.
- Require independent evidence and human approval before a consequential action such as a ban.
- Enforce permissions and moderation rules in application code; a prompt saying “do not ban” is not an access-control boundary.
- Treat model-generated summaries and verdicts as untrusted recommendations, and log their source evidence for review.

The challenge result confirms the intended impact in the Academy environment. It does not establish that the summarizer's identity claims were factually correct; the exercise demonstrates that the system accepted attacker-controlled instructions at a privileged workflow stage.
